The exact probability calculated using Variable Elimination is 0.5367, while rejection sampling with 10,000 accepted samples gives an estimate that should be close to this value.

The two results may differ slightly because rejection sampling uses random samples, and increasing the number of samples generally improves the estimate.

In [ ]:

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

# 1. Build the Bayesian Network
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel")
])

cpd_heatwave = TabularCPD(
    variable="heatwave", variable_card=2,
    values=[[0.3], [0.7]],
    state_names={"heatwave": ["yes", "no"]}
)

cpd_overload = TabularCPD(
    variable="overload", variable_card=2,
    values=[[0.6, 0.1], [0.4, 0.9]],
    evidence=["heatwave"], evidence_card=[2],
    state_names={
        "overload": ["yes", "no"],
        "heatwave": ["yes", "no"]
    }
)

cpd_powercut = TabularCPD(
    variable="powercut", variable_card=2,
    values=[
        [0.7, 0.2, 0.5, 0.05],
        [0.3, 0.8, 0.5, 0.95]
    ],
    evidence=["heatwave", "overload"],
    evidence_card=[2, 2],
    state_names={
        "powercut": ["yes", "no"],
        "heatwave": ["yes", "no"],
        "overload": ["yes", "no"]
    }
)

cpd_cancel = TabularCPD(
    variable="cancel", variable_card=2,
    values=[[0.8, 0.1], [0.2, 0.9]],
    evidence=["powercut"], evidence_card=[2],
    state_names={
        "cancel": ["yes", "no"],
        "powercut": ["yes", "no"]
    }
)

model.add_cpds(
    cpd_heatwave, cpd_overload,
    cpd_powercut, cpd_cancel
)

print("Model valid:", model.check_model())

# 2. Exact inference
inference = VariableElimination(model)

result = inference.query(
    variables=["heatwave"],
    evidence={"cancel": "yes"},
    show_progress=False
)

print(result)

exact = result.get_value(heatwave="yes")
print(f"Exact probability = {exact:.4f}")

# 3. Rejection sampling
sampler = BayesianModelSampling(model)

samples = sampler.rejection_sample(
    evidence=[State("cancel", "yes")],
    size=10000,
    show_progress=False
)

print("Number of accepted samples:", len(samples))

estimate = (samples["heatwave"] == "yes").mean()

print(f"Sampling estimate = {estimate:.4f}")


: 